# QPA Fidelity Decay — End-to-End Notebook

This notebook walks through the full QPA (Quantum Purification Algorithm) experiment pipeline step by step:

1. **Configuration** — Define experiment parameters
2. **Backend Setup** — Connect to IBM Quantum hardware (`ibm_marrakesh`)
3. **Circuit Generation** — Build unrolled circuits for all execution paths
4. **Noise Application** — Apply Pauli Twirling noise for each λ point
5. **Transpilation** — Compile circuits for the target hardware
6. **Submission** — Send circuits to IBM and collect results
7. **Result Processing** — Extract counts, filter by path conditions, compute fidelity
8. **Plotting** — Compare experimental fidelity decay against theory

**Experiments:**
- N=3, K=2, T=3 (3 registers, 2 qubits each, 3 trial rounds)
- N=5, K=2, T=3 (5 registers, 2 qubits each, 3 trial rounds)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from collections import defaultdict
from tqdm.notebook import tqdm

from core.circuit_factory import CircuitFactory
from core.noise_models import PauliTwirlingStrategy
from execution.backend_handler import IBMRuntimeHandler
from execution.transpiler_service import TranspilerService
from analysis.result_processor import ResultProcessor
from analysis.fidelity_calc import calculate_success_rate

from qiskit.circuit import CircuitInstruction
from qiskit.circuit.library import XGate, RZGate
from qiskit_ibm_runtime import SamplerV2 as IBMSampler

print("Imports loaded.")

ModuleNotFoundError: No module named 'matplotlib'

## Step 1: Configuration

Define the experiment parameters for both runs. We use the **unrolled** circuit strategy with **Pauli Twirling** noise on the real `ibm_marrakesh` backend.

| Parameter | Description | Value |
|-----------|-------------|-------|
| `N` | Number of registers (must be odd) | 3, 5 |
| `K` | Qubits per register (qudit dimension d=2^k) | 2 |
| `T` (n_trials) | Number of QPA trial rounds (circuit depth) | 3 |
| `POINTS` | Number of λ (noise) values to sweep | 25 |
| `LAMBDA_MIN/MAX` | Noise parameter range | 0.0 – 1.0 |
| `N_RANDOM` | Pauli twirling instances per λ point (ensemble size) | 50 |
| `SHOTS` | Total shots budget per λ point (split across instances) | 10000 |
| `BATCH_SIZE` | Circuits per IBM job submission | 50 |
| `DEVICE` | IBM Quantum backend | ibm_marrakesh |

In [ ]:
# ----- Experiment Parameters -----
K = 2                # Qubits per register (d = 2^K = 4)
T = 3                # Number of QPA trial rounds
POINTS = 25          # Number of lambda values in the sweep
LAMBDA_MIN = 0.0     # Minimum noise parameter
LAMBDA_MAX = 1.0     # Maximum noise parameter
N_RANDOM = 50        # Pauli twirling instances per circuit (ensemble size)
SHOTS = 10000        # Total shots budget per lambda point
BATCH_SIZE = 50      # Max circuits per job submission
DEVICE = "ibm_marrakesh"  # IBM Quantum backend
NO_RESET = False     # Reuse ancillas with reset (standard mode)

# The two experiments
EXPERIMENTS = [
    {"n": 3, "k": K, "t": T, "label": "N=3, K=2, T=3"},
    {"n": 5, "k": K, "t": T, "label": "N=5, K=2, T=3"},
]

# Derived
lambdas = np.linspace(LAMBDA_MIN, LAMBDA_MAX, POINTS)
shots_per_circuit = max(1, SHOTS // N_RANDOM)

print(f"Lambda sweep: {POINTS} points from {LAMBDA_MIN} to {LAMBDA_MAX}")
print(f"Shots per circuit instance: {shots_per_circuit}")
print(f"Pauli twirling instances per lambda: {N_RANDOM}")
print(f"Experiments: {[e['label'] for e in EXPERIMENTS]}")

## Step 2: Backend Setup

Connect to the real IBM Quantum backend `ibm_marrakesh` via `QiskitRuntimeService`. Authentication is loaded from the `.env` file (`IBM_QUANTUM_TOKEN` or `CRN`).

In [ ]:
# Initialize the IBM Runtime backend handler
backend_handler = IBMRuntimeHandler(backend_name=DEVICE)
backend = backend_handler.get_backend()

print(f"Backend: {backend.name}")
print(f"Number of qubits: {backend.num_qubits}")
print(f"Backend status: {backend.status()}")

## Step 3: Circuit Generation (Unrolled Strategy)

The **unrolled** strategy pre-computes all possible execution paths as separate static circuits. For each trial round, every pair of registers undergoes a Schur (swap) test. Each possible combination of pass/fail outcomes produces a distinct circuit branch.

- For N=3 (1 pair per trial): branches per trial = 2^1 = 2
- For N=5 (2 pairs per trial): branches per trial = 2^2 = 4

We first build **golden circuits** (no noise) to inspect the structure, then transpile them once for reuse.

In [ ]:
# Build golden (noiseless) circuits for each experiment
# These will be transpiled once and reused with noise injected later (fast path)

golden_data_per_exp = {}

for exp in EXPERIMENTS:
    n = exp["n"]
    label = exp["label"]
    
    # Create the unrolled strategy (no noise yet)
    strategy = CircuitFactory.create_strategy("unrolled", K, T, n, no_reset=NO_RESET)
    strategy.set_noise_strategy(None)
    
    # Build circuits with epsilon=0 (no noise)
    golden_data = strategy.build(0.0)
    
    golden_circuits = [item['circuit'] for item in golden_data]
    golden_metadata = [{k: v for k, v in item.items() if k != 'circuit'} for item in golden_data]
    
    golden_data_per_exp[n] = {
        'circuits': golden_circuits,
        'metadata': golden_metadata,
        'strategy': strategy
    }
    
    print(f"\n--- {label} ---")
    print(f"  Total unrolled paths (circuits): {len(golden_circuits)}")
    print(f"  Qubits per circuit: {golden_circuits[0].num_qubits}")
    print(f"  Classical bits per circuit: {golden_circuits[0].num_clbits}")
    
    # Show path conditions for first few circuits
    for i, meta in enumerate(golden_metadata[:4]):
        conds = meta.get('conditions', {})
        path_name = meta.get('metadata', {}).get('path_name', f'path_{i}')
        print(f"  {path_name}: conditions = {conds}")

### Visualize a sample circuit

Draw one of the golden circuits to see the Schur test / cyclic rotation structure.

In [ ]:
# Draw the first golden circuit for N=3 (all-success path)
sample_qc = golden_data_per_exp[3]['circuits'][0]
print(f"Circuit: {golden_data_per_exp[3]['metadata'][0]}")
sample_qc.draw('mpl', fold=120)

## Step 4: Transpilation

Transpile the golden circuits **once** for `ibm_marrakesh` at optimization level 3. These transpiled circuits are reused for every noise instance — we only inject Pauli gates at the beginning (fast path).

In [ ]:
# Transpile golden circuits for the real backend
transpiler = TranspilerService(backend, optimization_level=3)

transpiled_per_exp = {}

for exp in EXPERIMENTS:
    n = exp["n"]
    label = exp["label"]
    golden_circuits = golden_data_per_exp[n]['circuits']
    
    print(f"Transpiling {len(golden_circuits)} golden circuits for {label}...")
    transpiled = transpiler.transpile(golden_circuits)
    if not isinstance(transpiled, list):
        transpiled = [transpiled]
    
    transpiled_per_exp[n] = transpiled
    
    # Report circuit depth and gate counts after transpilation
    qc0 = transpiled[0]
    print(f"  Transpiled circuit 0: depth={qc0.depth()}, "
          f"gates={dict(qc0.count_ops())}")
    print(f"  Layout: {qc0.layout.initial_layout if qc0.layout else 'None'}")

print("\nAll golden circuits transpiled.")

## Step 5: Noise Application & Circuit Instance Generation

For each λ value and each Pauli twirling instance, we:
1. Copy the transpiled golden circuit
2. Generate random Pauli noise operations via `PauliTwirlingStrategy.generate_noise_ops()`
3. Insert them at the beginning of the circuit (mapped through the transpilation layout)

This is the **fast path** — no re-transpilation needed per instance.

We convert Pauli gates (Z, Y) to ISA-safe gates (RZ, X+RZ) for IBM hardware compatibility.

In [ ]:
def build_noisy_batch(transpiled_golden, golden_circuits, golden_metadata, epsilon, n_random, batch_size):
    """
    Build a batch of noisy circuit instances from pre-transpiled golden circuits.
    
    For each random instance, copies each golden circuit and injects Pauli noise
    at the beginning (fast path). Returns circuits + metadata ready for submission.
    
    Args:
        transpiled_golden: List of transpiled golden circuits
        golden_circuits: List of original (pre-transpile) golden circuits (for register info)
        golden_metadata: List of metadata dicts for each golden circuit
        epsilon: Noise strength (depolarizing probability)
        n_random: Number of random Pauli twirling instances
        batch_size: Max circuits per batch
        
    Returns:
        List of (batch_circuits, batch_metadata) tuples
    """
    all_batches = []
    batch_circuits = []
    batch_metadata = []
    
    for _ in range(n_random):
        noise_strategy = PauliTwirlingStrategy(K)
        
        for i, qc_transpiled in enumerate(transpiled_golden):
            qc_instance = qc_transpiled.copy()
            orig_qc = golden_circuits[i]
            
            # Get data registers from original circuit
            data_regs = [reg for reg in orig_qc.qregs if reg.name.startswith("R")]
            noise_ops = noise_strategy.generate_noise_ops(data_regs, epsilon)
            
            # Map noise ops through the transpilation layout
            layout = qc_transpiled.layout.initial_layout if qc_transpiled.layout else None
            
            for gate, logical_qubit in noise_ops:
                target_qubit = None
                if layout and logical_qubit in layout:
                    phys_qubit_idx = layout[logical_qubit]
                    target_qubit = qc_instance.qubits[phys_qubit_idx]
                elif logical_qubit in qc_instance.qubits:
                    target_qubit = logical_qubit
                else:
                    for q in qc_instance.qubits:
                        if hasattr(q, 'register') and hasattr(logical_qubit, 'register'):
                            if q.register.name == logical_qubit.register.name and q.index == logical_qubit.index:
                                target_qubit = q
                                break
                
                if target_qubit:
                    # Convert to ISA-safe gates for IBM hardware
                    if gate.name == 'z':
                        qc_instance.data.insert(0, CircuitInstruction(RZGate(np.pi), (target_qubit,), ()))
                    elif gate.name == 'y':
                        qc_instance.data.insert(0, CircuitInstruction(RZGate(np.pi), (target_qubit,), ()))
                        qc_instance.data.insert(0, CircuitInstruction(XGate(), (target_qubit,), ()))
                    else:
                        qc_instance.data.insert(0, CircuitInstruction(gate, (target_qubit,), ()))
            
            batch_circuits.append(qc_instance)
            batch_metadata.append(golden_metadata[i])
            
            # Split into batches
            if len(batch_circuits) >= batch_size:
                all_batches.append((batch_circuits, batch_metadata))
                batch_circuits = []
                batch_metadata = []
    
    # Final partial batch
    if batch_circuits:
        all_batches.append((batch_circuits, batch_metadata))
    
    return all_batches

# Quick test: build one batch for epsilon=0.5, N=3
test_batches = build_noisy_batch(
    transpiled_per_exp[3], 
    golden_data_per_exp[3]['circuits'],
    golden_data_per_exp[3]['metadata'],
    epsilon=0.5, n_random=2, batch_size=50
)
total_circuits = sum(len(b[0]) for b in test_batches)
print(f"Test: epsilon=0.5, N=3, 2 instances -> {total_circuits} circuits in {len(test_batches)} batch(es)")

## Step 6: Submission to IBM Quantum

For each experiment (N=3 and N=5), sweep over all λ values. At each λ:
1. Generate `N_RANDOM` noisy circuit instances (fast path)
2. Submit them in batches to `ibm_marrakesh`
3. Wait for results and store raw counts

**Note:** This step submits real jobs to IBM Quantum hardware. Each lambda point produces multiple batches. The total number of jobs = `POINTS × num_batches_per_lambda × 2 experiments`.

In [ ]:
def run_experiment(n, transpiled_golden, golden_circuits, golden_metadata, 
                   lambdas, n_random, shots_per_circuit, batch_size, backend):
    """
    Run the full lambda sweep for one experiment (one value of N).
    
    For each lambda value:
      1. Build noisy circuit batches (fast path)
      2. Submit to IBM via SamplerV2
      3. Collect results and compute fidelity
    
    Args:
        n: Number of registers
        transpiled_golden: Pre-transpiled golden circuits
        golden_circuits: Original golden circuits (for register info)
        golden_metadata: Metadata for each golden path
        lambdas: Array of noise parameter values
        n_random: Number of Pauli twirling instances
        shots_per_circuit: Shots per circuit instance
        batch_size: Max circuits per job
        backend: IBM backend object
        
    Returns:
        List of {'lambda': float, 'fidelity': float} dicts
    """
    result_processor = ResultProcessor(K)
    results = []
    
    sampler = IBMSampler(mode=backend)
    
    for epsilon in tqdm(lambdas, desc=f"N={n} lambda sweep"):
        # Build all noisy batches for this lambda
        batches = build_noisy_batch(
            transpiled_golden, golden_circuits, golden_metadata,
            epsilon, n_random, batch_size
        )
        
        # Accumulate stats across all batches
        global_path_stats = defaultdict(lambda: {'success': 0, 'total': 0})
        
        for batch_idx, (batch_circs, batch_meta) in enumerate(batches):
            # Submit batch
            pubs = [(qc, None, shots_per_circuit) for qc in batch_circs]
            
            try:
                job = sampler.run(pubs)
                job_id = job.job_id()
                
                # Wait for results
                pub_result = job.result()
                
                # Extract counts from each PUB result
                extracted_counts = ResultProcessor.extract_counts_from_job_result(pub_result, is_dynamic=False)
                
                # Determine total classical bits per circuit
                total_clbits_list = []
                for counts in extracted_counts:
                    if counts:
                        first_key = next(iter(counts))
                        total_clbits_list.append(len(first_key.replace(" ", "")))
                    else:
                        total_clbits_list.append(0)
                
                # Aggregate path statistics
                batch_stats = result_processor.aggregate_batch_stats(
                    extracted_counts, batch_meta, total_clbits_list
                )
                for cond_key, stats in batch_stats.items():
                    global_path_stats[cond_key]['success'] += stats['success']
                    global_path_stats[cond_key]['total'] += stats['total']
                    
            except Exception as e:
                print(f"  Error at lambda={epsilon:.4f}, batch {batch_idx}: {e}")
                import traceback
                traceback.print_exc()
        
        # Compute fidelity for this lambda point
        fidelity = 0.0
        for stats in global_path_stats.values():
            if stats['total'] > 0:
                fidelity += stats['success'] / stats['total']
        
        results.append({'lambda': epsilon, 'fidelity': fidelity})
        print(f"  N={n}, lambda={epsilon:.4f} -> fidelity={fidelity:.4f}")
    
    return results

print("run_experiment() defined.")

### Run Experiment: N=3, K=2, T=3

In [ ]:
results_n3 = run_experiment(
    n=3,
    transpiled_golden=transpiled_per_exp[3],
    golden_circuits=golden_data_per_exp[3]['circuits'],
    golden_metadata=golden_data_per_exp[3]['metadata'],
    lambdas=lambdas,
    n_random=N_RANDOM,
    shots_per_circuit=shots_per_circuit,
    batch_size=BATCH_SIZE,
    backend=backend
)

df_n3 = pd.DataFrame(results_n3)
print(f"\nN=3 results: {len(df_n3)} lambda points collected")
df_n3.head()

### Run Experiment: N=5, K=2, T=3

In [ ]:
results_n5 = run_experiment(
    n=5,
    transpiled_golden=transpiled_per_exp[5],
    golden_circuits=golden_data_per_exp[5]['circuits'],
    golden_metadata=golden_data_per_exp[5]['metadata'],
    lambdas=lambdas,
    n_random=N_RANDOM,
    shots_per_circuit=shots_per_circuit,
    batch_size=BATCH_SIZE,
    backend=backend
)

df_n5 = pd.DataFrame(results_n5)
print(f"\nN=5 results: {len(df_n5)} lambda points collected")
df_n5.head()

## Step 7: Save Results

Save the fidelity vs. lambda data to CSV files for reproducibility and later analysis.

In [ ]:


# Save results to CSV
results_dir = os.path.join("data", "results", "end_to_end")
os.makedirs(results_dir, exist_ok=True)

csv_n3 = os.path.join(results_dir, "results_n3_k2_t3_ibm_marrakesh.csv")
csv_n5 = os.path.join(results_dir, "results_n5_k2_t3_ibm_marrakesh.csv")

df_n3.to_csv(csv_n3, index=False)
df_n5.to_csv(csv_n5, index=False)

print(f"N=3 results saved to: {csv_n3}")
print(f"N=5 results saved to: {csv_n5}")

## Step 8: Plotting — Experimental vs. Theory

Plot the experimental fidelity decay curves for N=3 and N=5 alongside their theoretical predictions.

**Theoretical curves** (for K=2, d=4):
- N=3: $F(\lambda) = \frac{1}{8}(8 - 2\lambda - 7\lambda^2 + 3\lambda^3)$
- N=5: $F(\lambda) = \frac{1}{640}(640 - 96\lambda - 224\lambda^2 - 700\lambda^3 + 693\lambda^4 - 153\lambda^5)$

In [ ]:
# --- Theoretical Curves ---
def theory_curve(lam, n, k):
    """Analytical fidelity prediction for QPA with n registers, k qubits/register."""
    if k == 2:  # d=4
        if n == 3:
            return (1/8) * (8 - 2*lam - 7*lam**2 + 3*lam**3)
        elif n == 5:
            return (1/640) * (640 - 96*lam - 224*lam**2 - 700*lam**3 + 693*lam**4 - 153*lam**5)
    return None

# --- Plot ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'lines.linewidth': 2,
    'lines.markersize': 8
})

fig, ax = plt.subplots(figsize=(12, 8))

lam_fine = np.linspace(0, 1, 200)
colors = {'n3': '#1f77b4', 'n5': '#ff7f0e'}

# Theory curves (dashed lines)
theory_n3 = theory_curve(lam_fine, 3, 2)
theory_n5 = theory_curve(lam_fine, 5, 2)
ax.plot(lam_fine, theory_n3, '--', color=colors['n3'], linewidth=1.5, alpha=0.6, label='Theory (N=3, K=2)')
ax.plot(lam_fine, theory_n5, '--', color=colors['n5'], linewidth=1.5, alpha=0.6, label='Theory (N=5, K=2)')

# Experimental data (scatter with markers)
ax.scatter(df_n3['lambda'], df_n3['fidelity'], color=colors['n3'], marker='o', s=60, 
           zorder=5, alpha=0.8, edgecolors='white', linewidth=0.5,
           label=f'IBM Marrakesh (N=3, K=2, T=3)')
ax.scatter(df_n5['lambda'], df_n5['fidelity'], color=colors['n5'], marker='^', s=60,
           zorder=5, alpha=0.8, edgecolors='white', linewidth=0.5,
           label=f'IBM Marrakesh (N=5, K=2, T=3)')

# Random guess baseline (1/d = 1/4 for K=2)
ax.axhline(y=0.25, color='gray', linestyle=':', alpha=0.4, label='Random guess (1/d = 0.25)')

ax.set_title('QPA Fidelity Decay — IBM Marrakesh', fontweight='bold')
ax.set_xlabel(r'Depolarizing Noise Strength ($\lambda$)')
ax.set_ylabel('Purified Fidelity')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0.20, 1.02)
ax.legend(loc='best', frameon=True, shadow=True)
ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

plt.tight_layout()

# Save plot
plot_path = os.path.join(results_dir, "fidelity_decay_ibm_marrakesh.png")
plt.savefig(plot_path, dpi=300)
print(f"Plot saved to: {plot_path}")

plt.show()

## Step 9: Result Inspection

Inspect the per-lambda fidelity values and compare against theoretical predictions.

In [ ]:
# Compare experimental vs theoretical fidelity
print("=== N=3, K=2, T=3 ===")
print(f"{'Lambda':>8} {'Exp Fidelity':>14} {'Theory':>10} {'Delta':>10}")
print("-" * 46)
for _, row in df_n3.iterrows():
    lam = row['lambda']
    exp_f = row['fidelity']
    th_f = theory_curve(lam, 3, 2)
    delta = exp_f - th_f if th_f is not None else float('nan')
    print(f"{lam:8.4f} {exp_f:14.4f} {th_f:10.4f} {delta:+10.4f}")

print(f"\n=== N=5, K=2, T=3 ===")
print(f"{'Lambda':>8} {'Exp Fidelity':>14} {'Theory':>10} {'Delta':>10}")
print("-" * 46)
for _, row in df_n5.iterrows():
    lam = row['lambda']
    exp_f = row['fidelity']
    th_f = theory_curve(lam, 5, 2)
    delta = exp_f - th_f if th_f is not None else float('nan')
    print(f"{lam:8.4f} {exp_f:14.4f} {th_f:10.4f} {delta:+10.4f}")